# 1- Importing Libraries

In [13]:
! pip install pandas
! pip install numpy
! pip install scikit-learn
! pip install nltk
! pip install seaborn
! pip install imbalanced-learn
! pip install langchain
! pip install langchain-huggingface
! pip install sentence-transformers

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.8/570.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 24.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 33.1 MB/s eta 0:00:00 0:00:01
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl (447 kB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)


In [3]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import nltk
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from nltk.tokenize import sent_tokenize,word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from imblearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix
from langchain_huggingface import HuggingFaceEmbeddings
nltk.download("stopwords")
nltk.download("punkt_tab")
lemmatizer = WordNetLemmatizer()
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# 2- Loading Dataset

In [4]:
df = pd.read_csv("Reviews.csv")
df = df.sample(200000)

# 3- Understanding Dataset

In [93]:
df.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='str')

In [31]:
df.shape

(200000, 10)

In [ ]:
df.nunique()
df["Score"].unique()

array([5, 1, 4, 2, 3])

In [ ]:
df['HelpfulnessNumerator']

0         1
1         0
2         1
3         3
4         0
         ..
568449    0
568450    0
568451    2
568452    1
568453    0
Name: HelpfulnessNumerator, Length: 568454, dtype: int64

# 4- Preprocessing the Dataset

In [105]:
df.isnull().sum()

Id                         0
ProductId                  0
UserId                     0
ProfileName                6
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   12
Text                       0
dtype: int64

In [106]:
df.dropna(inplace=True)

In [96]:
df.duplicated().sum()

np.int64(0)

In [107]:
df.drop(['Id','UserId','Time','Summary','HelpfulnessNumerator','HelpfulnessDenominator','ProfileName'],axis=1,inplace=True)

In [10]:
df["Sentiment"] = df["Score"].apply(lambda x:"Negative" if x<=2 else ("Neutral" if x==3 else "Positive"))

In [109]:
df.drop("Score",axis=1,inplace=True)

# 5- Text Preprocessing Using NLP

In [7]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))  # load ONCE

text_preprocessed = []

for text in tqdm(df["Text"].astype(str)):  # ensure all rows are strings
    text = text.lower()
    text = re.sub(r'[^A-Za-z0-9]', ' ', text)
    
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words and len(word) >= 2]
    
    text_preprocessed.append(" ".join(words))

100%|██████████| 200000/200000 [01:10<00:00, 2826.23it/s]


# 6- Training Machine Learning Models Using Count Vectorizer as Technique

In [142]:
X=text_preprocessed
y = df["Sentiment"]

X_train, X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)


pipe = Pipeline([
    ('tfidf', CountVectorizer()),
    ('smote', SMOTE(random_state=42)),
    ('lgr', LogisticRegression(max_iter=2000))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

In [145]:
print(accuracy_score(y_test,y_pred)*100)
print(f1_score(y_test,y_pred,average="weighted")*100)
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

81.65513792816068
82.73454033229649
              precision    recall  f1-score   support

    Negative       0.64      0.71      0.67      8650
     Neutral       0.29      0.43      0.35      4494
    Positive       0.93      0.87      0.90     46851

    accuracy                           0.82     59995
   macro avg       0.62      0.67      0.64     59995
weighted avg       0.84      0.82      0.83     59995

[[ 6111  1089  1450]
 [  975  1915  1604]
 [ 2400  3488 40963]]


# 7- Training Machine Learning Models Using LLM Embeddings as Technique

In [9]:
df

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
120419,120420,B005K4Q37A,A2G5994VXZSPLI,TLB,0,0,5,1340496000,These taste so good!,"I love this coffee. It is wonderful, smoothe a..."
392099,392100,B0013ES5WC,AWNOWP4QGRXK3,"P. Ju ""Scootrat""",0,3,4,1247616000,It filters,"It does its job. Pretty expensive, at $3-$4 a ..."
392059,392060,B001RV8CHY,A154TLVB3GAIU7,Diane R. Catanzaro,0,0,4,1348185600,Excellent on fish!,"Excellent for grilling, frying, or baking fish..."
416153,416154,B002HFWMOI,AJ3QAM3XX4BUE,"Barefootokie75 ""Sarah Varadi Miller""",0,0,5,1306713600,Happy,I've been using Nestle's Fat Free Hot Cocoa Mi...
369143,369144,B000FNEZGM,A3KQUWSBYUQGOI,J. Trellu,0,0,5,1219968000,Our daughter LOVES these cookie bars!,"Our daughter has several food allergies, so Na..."
...,...,...,...,...,...,...,...,...,...,...
65187,65188,B004U8WODY,A30J8KM3H9TDBU,"Clasina Gm Oneill ""Maria""",0,0,4,1333584000,Lovely with tea.,"These are good digestives, crunchy, not too sw..."
517024,517025,B000G6L2ZU,A2T2MJT8LFLBXH,"Clow Angel ""my thoughts, my memories.""",2,2,5,1184976000,"Cheap, Well Worth It.","Now, I've always been a fan of gummies. I love..."
28396,28397,B007OXJMD2,A30GEP1I3ZJABG,J. Thoman,0,0,4,1326153600,"Good, but a little pricey","I enjoy a cup of hazelnut coffee on occasion, ..."
247697,247698,B001E6GFZI,AZQ05PT95RA5H,W. Mattia,4,5,5,1196294400,"Must have, no joke",I do not think I can go a day without it. It ...


In [14]:
X= text_preprocessed
y = df["Sentiment"]

embed = HuggingFaceEmbeddings(model="microsoft/harrier-oss-v1-0.6b")
embedded  = embed.embed_documents(X)

X_train, X_test,y_train,y_test = train_test_split(embedded,y,test_size=0.3,random_state=42)


pipe = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('lgr', LogisticRegression(max_iter=2000))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

/Users/prabhsandhu/Downloads/Machine_Learning_Project/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/prabhsandhu/Downloads/Machine_Learning_Project/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:722: UserWarning: Not enough free disk space to download the file. The expected file size is: 1192.13 MB. The target location /Users/prabhsandhu/.cache/huggingface/hub/models--microsoft--harrier-oss-v1-0.6b/blobs only has 1073.95 MB free disk space.
  warnings.warn(


OSError: Can't load the model for 'microsoft/harrier-oss-v1-0.6b'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'microsoft/harrier-oss-v1-0.6b' is the correct path to a directory containing a file named pytorch_model.bin.

In [ ]:
print(accuracy_score(y_test,y_pred)*100)
print(f1_score(y_test,y_pred,average="weighted")*100)
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

87.59563296941411
87.93733173288143
              precision    recall  f1-score   support

    Negative       0.74      0.80      0.77      8650
     Neutral       0.47      0.53      0.50      4494
    Positive       0.95      0.92      0.94     46851

    accuracy                           0.88     59995
   macro avg       0.72      0.75      0.73     59995
weighted avg       0.88      0.88      0.88     59995

[[ 6909   747   994]
 [  889  2388  1217]
 [ 1595  2000 43256]]
